# Storage facility SCADA — the WEM battery fleet

`FacilityScada-*` is the per-facility 5-minute dispatch record. This notebook takes the storage
subset: seven grid-scale batteries, 1,096,453 facility-interval rows across 307,008 five-minute
intervals, 1 October 2023 to 1 September 2026. It is the notebook the price analysis handed its open
question to — *did the storage fleet cause the price
flattening, or merely arrive alongside it?* — and it answers a second one the registry cannot: what
these machines actually are.

Three findings run through it:

- **The fleet went from one battery to seven, and from 4.7 GWh to 837 GWh discharged a year.** Every
  facility charges in the solar trough and discharges into the evening peak, and the average Winter
  day's swing deepened from 43 MW peak-to-trough to 1,075 MW between 2024 and 2026.
- **The registry does not say how big these batteries are, but the dispatch does.** Reconstructing
  state of charge as the running integral of dispatch recovers an energy rating for each unit, and
  with it a duration: Kwinana ESR1 is a **2-hour** battery, the other six are **4-hour**. Round-trip
  efficiency lands at **86-88%** fleet-wide, with no detectable degradation over the three years
  Kwinana ESR1 has been running.
- **The arbitrage that justified building them has largely gone.** Spread capture fell from
  123 to 20 $/MWh between 2024 and 2026 on a like-for-like January-August basis, and energy-only
  gross margin peaked in 2025. In 2026 **five of the seven units lose money on energy arbitrage
  alone**.

That last result carries a caveat, stated once here and again at the end: **this is energy arbitrage
only.** WEM batteries also earn capacity credits and essential-system-service payments, neither of
which is in this data. A negative energy margin is not an unprofitable battery. What it does show is
that the *energy* case has thinned to nothing, and the other revenue streams are now carrying the
asset.

Two structural quirks shape every number below, both from `docs/DATA_NOTES.md`:

- **`Average MWh` is energy per 5-minute interval, not MW** (quirk 6). Multiply by 12. The loader
  exposes an explicit `mw` column, and this notebook uses it everywhere.
- **`System Size (MW)` is the two-sided span**, maximum charge plus maximum discharge (quirk 7).
  Collie BESS2 is a 300 MW battery registered as 600. Power figures here are measured from dispatch,
  never taken from the registry.

In [1]:
# Load storage SCADA and the facility registry
import sys
sys.path.insert(0, '../src')  # Add src directory to path (go up one level from notebooks/)
from wa_data import (load_facility_scada, load_facilities, load_price,
                     load_demand, add_time_parts)

# Load with explicit path (go up one level from notebooks/)
s = load_facility_scada(raw='../data/raw')
fac = load_facilities(raw='../data/raw')

In [2]:
print(s.head())
print(s.shape)
print(s.columns.tolist())

                   ts facility_code  Average MWh     mw
0 2023-10-01 08:00:00  KWINANA_ESR1       -0.057 -0.684
1 2023-10-01 08:05:00  KWINANA_ESR1       -0.061 -0.732
2 2023-10-01 08:10:00  KWINANA_ESR1       -0.047 -0.564
3 2023-10-01 08:15:00  KWINANA_ESR1        0.014  0.168
4 2023-10-01 08:20:00  KWINANA_ESR1       -0.005 -0.060
(1096453, 4)
['ts', 'facility_code', 'Average MWh', 'mw']


In [3]:
# The two unit quirks, checked rather than trusted. `Average MWh` is energy per
# 5-minute interval: if that is right, mw / Average MWh is exactly 12 everywhere,
# and the measured two-sided span matches the registered System Size.
import numpy as np
import pandas as pd

nz = s[s['Average MWh'] != 0]
print(f"mw / Average MWh: min {(nz.mw / nz['Average MWh']).min():.6f}  "
      f"max {(nz.mw / nz['Average MWh']).max():.6f}   (expect exactly 12)")

span = s.groupby('facility_code').mw.agg(max_discharge='max', max_charge='min')
span['measured_span'] = span.max_discharge - span.max_charge
span = span.join(fac.set_index('facility_code')['system_size_mw'])
span['ratio'] = span.measured_span / span.system_size_mw
print("\nmeasured two-sided span vs registered System Size (MW):")
print(span.round(1).to_string())
print("\nALINTA_WGP_ESR1 is the one mismatch: it began dispatching 25 July 2026,")
print("five weeks before the record ends, and is still ramping.")

mw / Average MWh: min 12.000000  max 12.000000   (expect exactly 12)

measured two-sided span vs registered System Size (MW):
                 max_discharge  max_charge  measured_span  system_size_mw  ratio
facility_code                                                                   
ALINTA_WGP_ESR1           25.2       -25.4           50.7           200.0    0.3
COLLIE_BESS2             300.1      -300.0          600.0           600.0    1.0
COLLIE_ESR1              200.0      -200.1          400.1           400.0    1.0
COLLIE_ESR4              252.0      -252.3          504.3           500.0    1.0
COLLIE_ESR5              252.7      -252.5          505.2           500.0    1.0
KWINANA_ESR1             100.7      -100.1          200.8           200.0    1.0
KWINANA_ESR2             226.4      -223.7          450.0           450.0    1.0

ALINTA_WGP_ESR1 is the one mismatch: it began dispatching 25 July 2026,
five weeks before the record ends, and is still ramping.


In [4]:
# Completeness. The expected grid is 5-minute, but the FACILITY count changes over
# the record as units commission, so a single expected-row count would be wrong.
# Completeness is therefore checked on the INTERVAL grid, not on rows.
iv = pd.Series(sorted(s.ts.unique()))
expected = int((iv.max() - iv.min()) / pd.Timedelta('5min')) + 1
print(f'coverage: {iv.min()}  ->  {iv.max()}')
print(f'5-min intervals: {len(iv):,} of {expected:,} expected  ->  {expected - len(iv)} missing')
print(f'duplicated (ts, facility) pairs: {s.duplicated(subset=["ts", "facility_code"]).sum()}')
print(f'rows: {len(s):,}   facilities: {s.facility_code.nunique()}')

# File-boundary quirk (1): monthly files are cut at 00:00 UTC = 08:00 AWST, so the
# November file carries the first 8 hours of 1 December. The loader concatenates
# and de-duplicates, so every slice below is taken by LOCAL DATE, never per file.
# 1 December 2024 is the boundary case: it is complete here, but it is split
# across bess-2024-11.csv and bess-2024-12.csv on disk.
b = s[(s.ts >= '2024-11-30') & (s.ts < '2024-12-03')]
print("\nintervals per local date across a file boundary (expect 288 each):")
print(b.groupby([b.ts.dt.date, b.facility_code]).size().unstack().to_string())

coverage: 2023-10-01 08:00:00  ->  2026-09-01 07:55:00
5-min intervals: 307,008 of 307,008 expected  ->  0 missing
duplicated (ts, facility) pairs: 0
rows: 1,096,453   facilities: 7

intervals per local date across a file boundary (expect 288 each):
facility_code  COLLIE_ESR1  KWINANA_ESR1  KWINANA_ESR2
ts                                                    
2024-11-30             288           288           288
2024-12-01             288           288           288
2024-12-02             288           288           288


## Visualisation

Six views, ordered fleet -> machine -> behaviour -> money:

1. **The fleet, and what it does with its time** — commissioning, installed power, and the three
   distinct kinds of "almost zero" a battery reports.
2. **Every interval, every facility** — carpet plots, charge and discharge on one diverging scale.
3. **What the registry does not say** — energy capacity and duration, reconstructed from dispatch.
4. **Round-trip efficiency** — with the error bound that makes the estimate honest.
5. **The daily cycle, deepening** — the average day by season and year.
6. **Spread capture and the margin collapse** — what the fleet buys at, sells at, and keeps.

Facility identity is carried by **position** — one panel or one row per facility — rather than by
seven cycled hues, which no categorical palette supports at all-pairs separation. Where colour does
carry a category it is the **site**, Collie / Kwinana / Alinta, which is three and validates.

In [5]:
# ── Chart setup: palette, shared theme, and derived frames ───────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chart chrome, light surface — identical to the demand, DPV and price notebooks,
# so all four read as one system.
SURFACE, INK, INK_2, MUTED, GRID, AXIS = (
    "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7")

BLUE, ORANGE, RED, AQUA = "#2a78d6", "#eb6834", "#e34948", "#1baf7a"

# SIGN is the semantic axis of this notebook, so it takes the DIVERGING pair:
# orange = charging, keeping the meaning it carries in the demand and price
# notebooks, blue = discharging, and a NEUTRAL GRAY midpoint — never a hue — so
# that "doing nothing" reads as nothing. Validated as a pair on this surface:
# worst CVD deltaE 24.7 (protan), normal-vision deltaE 33.6, both clear.
GRAY_MID = "#f0efec"
ORANGE_RAMP = ["#fbe0d4", "#f7c0a3", "#f39d73", "#eb6834", "#c94e1f", "#a03c17"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95"]
DIVERGING = [[i / 12, c] for i, c in
             enumerate(ORANGE_RAMP[::-1] + [GRAY_MID] + BLUE_RAMP)]

# SITE is the only categorical dimension here — three values, which is what a
# categorical palette can carry at all-pairs separation (the first three slots of
# the reference theme; validated: worst all-pairs CVD deltaE 9.2, normal 24.0).
# Aqua sits below 3:1 on this surface, so site series are DIRECT-LABELLED — the
# relief a contrast warning obliges, never colour alone.
SITE_COLOR = {"Collie": BLUE, "Kwinana": ORANGE, "Alinta": AQUA}
SITES = ["Collie", "Kwinana", "Alinta"]


def site_of(f):
    return "Collie" if f.startswith("COLLIE") else (
        "Kwinana" if f.startswith("KWINANA") else "Alinta")


# YEAR is ordinal, so it keeps the one-hue ramp from the price notebook: the later
# the year, the darker the line. Colour follows the year, never its rank.
YEAR_COLOR = {2023: "#86b6ef", 2024: "#3987e5", 2025: "#256abf", 2026: "#0d366b"}
PARTIAL = {2023: "Oct-Dec only", 2026: "to 1 Sep"}
YEARS = [2023, 2024, 2025, 2026]
LABEL = {y: (f"{y} ({PARTIAL[y]})" if y in PARTIAL else str(y)) for y in YEARS}
DASH = {y: ("dot" if y in PARTIAL else "solid") for y in YEARS}


def style(fig, title, subtitle=None, height=420, hover="x unified",
          top=108, bottom=58, legend_y=1.0, showlegend=True):
    """Shared theme: light surface, recessive grid, muted axes, ink-coloured text."""
    head = f"<b>{title}</b>"
    if subtitle:
        head += f"<br><span style='font-size:12.5px;color:{INK_2}'>{subtitle}</span>"
    fig.update_layout(
        title=dict(text=head, font=dict(size=17, color=INK), x=0, xanchor="left", y=0.97),
        paper_bgcolor=SURFACE, plot_bgcolor=SURFACE, hovermode=hover,
        font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=12, color=INK_2),
        height=height, margin=dict(t=top, r=30, b=bottom, l=74), showlegend=showlegend,
        legend=dict(orientation="h", yanchor="bottom", y=legend_y, xanchor="left", x=0,
                    bgcolor="rgba(0,0,0,0)", font=dict(size=11.5)))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS,
                     ticks="outside", tickcolor=AXIS, tickfont=dict(color=MUTED))
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS,
                     ticks="outside", tickcolor=AXIS, tickfont=dict(color=MUTED))
    for a in (fig.layout.annotations or []):
        if a.font is None or a.font.size is None:
            a.font = dict(size=12.5, color=INK_2)
    return fig


# ── derived frames used by more than one chart ───────────────────────────────
s = s.copy()
s["site"] = s.facility_code.map(site_of)
s["date"] = s.ts.dt.normalize()
s["ym"] = s.ts.dt.to_period("M").dt.to_timestamp()
_e = s["Average MWh"]
s["chg"] = np.where(_e < 0, -_e, 0.0)    # energy INTO the battery, MWh, positive
s["dis"] = np.where(_e > 0, _e, 0.0)     # energy OUT of the battery, MWh, positive

# "Dispatch" means |MW| > 0.5 — above the parasitic band, below any real setpoint.
DISPATCH_MW = 0.5
first = s[s.mw.abs() > DISPATCH_MW].groupby("facility_code").ts.min().sort_values()
FACS = list(first.index)                 # commissioning order, used by every chart
PMAX = s.groupby("facility_code").mw.max()          # measured discharge power, MW

# A facility is MATURE from the first interval it reaches 90% of its eventual peak
# power. Before that it is commissioning, and its efficiency and capacity figures
# describe a test programme rather than a battery.
mature = {f: s.loc[(s.facility_code == f) & (s.mw >= 0.9 * PMAX[f]), "ts"].min()
          for f in FACS}

# Drop any month that is not fully covered, so monthly series are comparable.
_mn = s.groupby("ym").ts.nunique()
FULL_MONTHS = sorted(_mn[_mn >= 0.98 * _mn.index.days_in_month * 288].index)

print(f"{len(FACS)} facilities, in commissioning order:")
for f in FACS:
    print(f"  {f:<16} first dispatch {first[f]:%Y-%m-%d}   mature {mature[f]:%Y-%m-%d}"
          f"   {PMAX[f]:6.1f} MW discharge   site {site_of(f)}")
print(f"\nfull months: {len(FULL_MONTHS)}  ({FULL_MONTHS[0]:%b %Y} to {FULL_MONTHS[-1]:%b %Y})")

7 facilities, in commissioning order:
  KWINANA_ESR1     first dispatch 2023-10-01   mature 2023-10-10    100.7 MW discharge   site Kwinana
  COLLIE_ESR1      first dispatch 2024-07-29   mature 2024-09-10    200.0 MW discharge   site Collie
  KWINANA_ESR2     first dispatch 2024-10-19   mature 2024-12-11    226.4 MW discharge   site Kwinana
  COLLIE_BESS2     first dispatch 2025-04-09   mature 2025-06-17    300.1 MW discharge   site Collie
  COLLIE_ESR4      first dispatch 2025-10-14   mature 2025-11-06    252.0 MW discharge   site Collie
  COLLIE_ESR5      first dispatch 2025-11-04   mature 2025-12-04    252.7 MW discharge   site Collie
  ALINTA_WGP_ESR1  first dispatch 2026-07-25   mature 2026-08-05     25.2 MW discharge   site Alinta

full months: 35  (Oct 2023 to Aug 2026)


### 1. The fleet, and what it does with its time

The upper panel is installed **discharge** power, measured from dispatch rather than read off the
registry, stacked by site and stepped at each commissioning date. It goes from 100 MW to about
1,357 MW in under three years, and almost all of the growth is at Collie.

The lower panel is what the fleet did with that power: energy discharged per month. Capacity steps;
throughput ramps behind it. The two panels share a time axis and are drawn separately rather than on
twin y-scales, because MW and GWh have no common scale and overlaying them would let the choice of
axis manufacture whatever relationship the reader was looking for.

The table underneath decomposes each facility's time into four states, and the distinction matters
for everything that follows. A battery reporting exactly `0.000` is **offline**. A battery reporting
a small negative value is **idle**, drawing auxiliary power for cooling and inverter standby. Only
`|MW| > 0.5` is real dispatch. Treating idle draw as charging is the easiest way to understate
round-trip efficiency, which is why chart 4 reports the figure both ways.

In [6]:
# ── Chart 1: fleet buildout and monthly throughput ───────────────────────────
days = pd.date_range(s.ts.min().normalize(), s.ts.max().normalize(), freq="1D")
step = pd.DataFrame(index=days)
for f in FACS:
    step[site_of(f)] = step.get(site_of(f), 0.0) + np.where(days >= first[f], PMAX[f], 0.0)
step = step[SITES]

thr = (s[s.ym.isin(FULL_MONTHS)].groupby("ym").dis.sum() / 1e3).rename("GWh")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    row_heights=[0.55, 0.45],
                    subplot_titles=["Installed discharge power, measured from dispatch",
                                    "Energy discharged per month"])

cum = 0.0
for site in SITES:
    fig.add_trace(go.Scatter(
        x=step.index, y=step[site], name=site, stackgroup="one", mode="lines",
        line=dict(width=1.4, color=SITE_COLOR[site]),
        hovertemplate="%{y:,.0f} MW" f"<extra>{site}</extra>"), row=1, col=1)
    # Direct label at the right edge, centred in the band: identity never rests on
    # the fill colour alone. Label text stays in ink, not the series hue.
    lo, hi = cum, cum + step[site].iloc[-1]
    if hi - lo > 40:
        fig.add_annotation(x=step.index[-1], y=(lo + hi) / 2, text=site, showarrow=False,
                           xanchor="right", xshift=-8, font=dict(size=11.5, color=INK_2),
                           row=1, col=1)
    cum = hi

fig.add_trace(go.Scatter(x=thr.index, y=thr.values, mode="lines", showlegend=False,
                         line=dict(color=BLUE, width=2), fill="tozeroy",
                         fillcolor="rgba(42,120,214,0.14)",
                         hovertemplate="%{y:,.0f} GWh<extra>discharged</extra>"), row=2, col=1)

style(fig, "One battery became seven in under three years",
      "Installed discharge power by site, and the energy the fleet actually moved. "
      "Separate panels on one shared time axis — never a second y-axis.",
      height=620, top=152, legend_y=1.10)
fig.update_yaxes(title_text="MW", row=1, col=1)
fig.update_yaxes(title_text="GWh per month", row=2, col=1)
fig.show()

st = s.assign(state=np.select(
    [s["Average MWh"] == 0, s.mw.abs() <= DISPATCH_MW, s.mw < 0],
    ["offline", "idle (auxiliary)", "charging"], "discharging"))
tab = (st.pivot_table(index="facility_code", columns="state", values="ts",
                      aggfunc="size", fill_value=0).reindex(FACS))
tab = 100 * tab.div(tab.sum(axis=1), axis=0)
print("share of each facility's reported intervals, by state (%):")
print(tab[["offline", "idle (auxiliary)", "charging", "discharging"]].round(1).to_string())
print("\nKWINANA_ESR1 and ESR2 idle about 13% of the time; the Collie units 25-36%.")
print("ALINTA_WGP_ESR1 is 90% offline-or-idle because it is five weeks old.")

share of each facility's reported intervals, by state (%):
state            offline  idle (auxiliary)  charging  discharging
facility_code                                                    
KWINANA_ESR1        11.4              13.6      35.2         39.9
COLLIE_ESR1         10.0              27.4      31.0         31.6
KWINANA_ESR2        12.9              13.3      33.6         40.2
COLLIE_BESS2        12.7              25.3      34.2         27.8
COLLIE_ESR4          9.5              36.0      27.4         27.1
COLLIE_ESR5         13.8              29.4      28.6         28.3
ALINTA_WGP_ESR1     58.5              31.2       7.5          2.8

KWINANA_ESR1 and ESR2 idle about 13% of the time; the Collie units 25-36%.
ALINTA_WGP_ESR1 is 90% offline-or-idle because it is five weeks old.


### 2. Every interval, every facility

One panel per facility, one column per day, one row per half hour, colour is dispatch — orange
charging, blue discharging, the chart surface itself for idle. The whole record is in view at once,
so commissioning, outages and the daily rhythm all read off the same picture.

Colour is **normalised to each facility's own peak power**, not shared in MW. The absolute scale is
already in chart 1, and a shared MW scale would render the 25 MW Alinta unit and the 100 MW Kwinana
ESR1 as blank panels next to the 300 MW Collie units. Normalising asks the question this chart is
actually for — *what does each machine do with itself* — and makes the seven answers comparable.

The pattern is the same everywhere and it sharpens over time: a broad orange band through the middle
of the day, a narrow dark blue band at the evening peak, and a second, weaker blue band before dawn.
The flat grey region at the left of each panel is not missing data: a unit appears in `FacilityScada`
and reports `0.000` for weeks before it first dispatches, and grey is the midpoint of the scale.

In [7]:
# ── Chart 2: dispatch carpets, one panel per facility ────────────────────────
# Aggregated to 30 minutes: at 5-minute resolution these are 288 x 1067 cells per
# facility and the browser pays for detail no eye can resolve at this size.
car = s.assign(hh=s.ts.dt.floor("30min"))
car = car.groupby(["facility_code", "hh"]).mw.mean().reset_index()
car["d"] = car.hh.dt.normalize()
car["h"] = car.hh.dt.hour + car.hh.dt.minute / 60

ROWS, COLS = 4, 2
fig = make_subplots(rows=ROWS, cols=COLS, shared_xaxes=False, shared_yaxes=True,
                    vertical_spacing=0.085, horizontal_spacing=0.05,
                    subplot_titles=[f"{f}  ·  {PMAX[f]:.0f} MW" for f in FACS])

for i, f in enumerate(FACS):
    r, c = i // COLS + 1, i % COLS + 1
    g = car[car.facility_code == f]
    piv = g.pivot_table(index="h", columns="d", values="mw", aggfunc="mean").sort_index()
    z = piv.values / PMAX[f] * 100                     # % of this unit's own peak
    fig.add_trace(go.Heatmap(
        z=z, x=piv.columns, y=piv.index, colorscale=DIVERGING, zmid=0,
        zmin=-100, zmax=100, zsmooth=False, showscale=(i == 0),
        colorbar=dict(title=dict(text="% of peak<br>power", side="top"), outlinewidth=0,
                      thickness=12, len=0.42, y=0.80, yanchor="top",
                      tickfont=dict(color=MUTED), ticksuffix="%"),
        hovertemplate="%{x|%d %b %Y} · %{y:.1f}h<br>%{z:.0f}% of peak"
                      f"<extra>{f}</extra>"), row=r, col=c)

style(fig, "Charge in the middle of the day, discharge into the evening peak",
      "Every half hour of the record, by facility. Orange is charging, blue is "
      "discharging, scaled to each unit's own peak power.",
      height=1320, hover="closest", top=126, showlegend=False)
fig.update_yaxes(tickvals=[0, 6, 12, 18, 24], range=[0, 24], gridcolor="rgba(0,0,0,0)")
fig.update_xaxes(showgrid=False, tickformat="%b<br>%Y", nticks=6)
for i in range(1, ROWS + 1):
    fig.update_yaxes(title_text="hour", row=i, col=1)
# The 8th cell of a 4x2 grid has no facility; leave it empty rather than stretching
# the layout, so every panel keeps the same aspect ratio and stays comparable.
fig.update_xaxes(visible=False, row=4, col=2)
fig.update_yaxes(visible=False, row=4, col=2)
fig.show()

### 3. What the registry does not say: energy capacity and duration

`facilities.csv` gives a `System Size (MW)`, and quirk 7 established that even that is the two-sided
span rather than the power rating. It gives **no energy rating at all** — and a battery's energy
rating, not its power, is what decides whether it can cover an evening peak.

The dispatch record contains it. Charge and discharge are the derivative of stored energy, so the
running integral of `-Average MWh` over a day is state of charge up to an unknown constant. The
constant cancels in the **swing**, the day's maximum minus its minimum, which is the energy the unit
actually moved between its emptiest and fullest moments. A typical day does not use the whole
battery — Collie ESR1's median day swings 497 MWh against a 95th-percentile 748 — so the **95th
percentile** is the statistic to read, and even that is a lower bound: it is the most the unit was
ever asked for, which need not be all it has. The grey bar on the right-hand panel runs back to the
median day, so the gap between typical and hardest use is visible rather than hidden.

The left panel shows the reconstruction for one facility on one day, so the method is visible rather
than asserted. The right panel divides each unit's 95th-percentile swing by its measured power to get
**duration**, and the answer is unambiguous: Kwinana ESR1 sits on the 2-hour line at 2.16 h, and the
five other commissioned units cluster on the 4-hour line, 3.74 to 4.14 h. Alinta is excluded — five
weeks of ramping is not a sample.

In [8]:
# ── Chart 3: state-of-charge reconstruction, and inferred duration ───────────
def swing(g):
    """Day's usable energy movement, MWh: max minus min of running stored energy."""
    c = (-g["Average MWh"]).cumsum()
    return c.max() - c.min()


sw = {}
for f in FACS:
    g = s[(s.facility_code == f) & (s.ts >= mature[f])].sort_values("ts")
    sw[f] = g.groupby("date").apply(swing, include_groups=False)

cap = pd.DataFrame({
    "p50": {f: v.median() for f, v in sw.items()},
    "p95": {f: v.quantile(0.95) for f, v in sw.items()},
    "days": {f: len(v) for f, v in sw.items()},
}).reindex(FACS)
cap["mw"] = PMAX.reindex(FACS)
cap["hours"] = cap.p95 / cap.mw
cap["h50"] = cap.p50 / cap.mw

EX_F, EX_D = "COLLIE_ESR1", pd.Timestamp("2025-12-30")   # a 95th-percentile day, full cycle
ex = s[(s.facility_code == EX_F) & (s.date == EX_D)].sort_values("ts")
soc = (-ex["Average MWh"]).cumsum()
soc = soc - soc.min()

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11, column_widths=[0.52, 0.48],
                    subplot_titles=[f"Reconstructed stored energy — {EX_F}, {EX_D:%d %b %Y}",
                                    "Inferred duration, all mature units"])

hrs = ex.ts.dt.hour + ex.ts.dt.minute / 60
fig.add_trace(go.Scatter(
    x=hrs, y=soc, mode="lines", line=dict(color=BLUE, width=2), showlegend=False,
    fill="tozeroy", fillcolor="rgba(42,120,214,0.12)",
    hovertemplate="%{x:.1f}h · %{y:,.0f} MWh above the day's minimum<extra></extra>"),
    row=1, col=1)
fig.add_hline(y=float(soc.max()), line=dict(color=MUTED, width=1, dash="dash"), row=1, col=1)
fig.add_annotation(x=0.3, y=float(soc.max()), text=f"swing {soc.max():,.0f} MWh",
                   showarrow=False, yshift=12, xanchor="left",
                   font=dict(size=11, color=MUTED), row=1, col=1)

d = cap.dropna(subset=["hours"]).drop(index="ALINTA_WGP_ESR1", errors="ignore")
order = d.sort_values("hours").index.tolist()
for f in order:
    fig.add_trace(go.Scatter(
        x=[d.loc[f, "h50"], d.loc[f, "hours"]], y=[f, f], mode="lines",
        line=dict(color=GRID, width=6), showlegend=False, hoverinfo="skip"), row=1, col=2)
fig.add_trace(go.Scatter(
    x=d.loc[order, "hours"], y=order, mode="markers", showlegend=False,
    marker=dict(size=11, color=[SITE_COLOR[site_of(f)] for f in order],
                line=dict(color=SURFACE, width=2)),
    hovertemplate="%{y}<br>%{x:.2f} h at rated power<extra></extra>"), row=1, col=2)
for h, lab in ((2, "2-hour"), (4, "4-hour")):
    fig.add_vline(x=h, line=dict(color=AXIS, width=1, dash="dot"), row=1, col=2)
    fig.add_annotation(x=h, y=1.0, yref="y2 domain", text=lab, showarrow=False,
                       yshift=8, font=dict(size=11, color=MUTED), row=1, col=2)

style(fig, "The dispatch record contains an energy rating the registry does not",
      "Left: stored energy as the running integral of dispatch. Right: 95th-percentile "
      "daily swing divided by measured power; the bar runs back to the median day.",
      height=470, hover="closest", top=132, showlegend=False)
fig.update_xaxes(title_text="hour", tickvals=[0, 6, 12, 18, 24], range=[0, 24], row=1, col=1)
fig.update_yaxes(title_text="MWh above the day's minimum", row=1, col=1)
fig.update_xaxes(title_text="hours at rated power", range=[0, 5], row=1, col=2)
fig.update_yaxes(categoryorder="array", categoryarray=order, row=1, col=2)
fig.show()

print("inferred usable energy capacity and duration (mature days only):")
print(cap[["days", "p50", "p95", "mw", "hours"]].rename(columns={
    "p50": "median swing MWh", "p95": "p95 swing MWh", "mw": "measured MW",
    "hours": "duration h"}).round(2).to_string())

inferred usable energy capacity and duration (mature days only):
                 days  median swing MWh  p95 swing MWh  measured MW  duration h
KWINANA_ESR1     1058            160.45         217.82       100.67        2.16
COLLIE_ESR1       715            497.35         747.77       199.98        3.74
KWINANA_ESR2      630            570.21         890.28       226.36        3.93
COLLIE_BESS2      442            733.97        1189.58       300.06        3.96
COLLIE_ESR4       300            661.81        1043.03       252.00        4.14
COLLIE_ESR5       272            704.10        1033.99       252.70        4.09
ALINTA_WGP_ESR1    28             25.58          64.96        25.25        2.57


### 4. Round-trip efficiency

Energy in, energy out. The ratio is the single most useful number about a battery, and it is
measurable here — but only if two traps are avoided, and the trap is more interesting than the
number.

**The first trap is the window.** Efficiency is discharge divided by charge over a period that
*begins and ends at the same state of charge*, and state of charge is not observed. Over a short
window the unknown boundary state swamps the answer: computed month by month, three of the seven
units return figures above 1.0 — one as high as 1.82 — which is not a battery. The error is bounded
by capacity divided by throughput, so it shrinks as the window grows: over each unit's full mature
record it is **under 0.5 percentage points**, and the estimate becomes sound. The bound is drawn on the chart as the bar through each
point; where you cannot see it, it is narrower than the marker.

Tempting and wrong: filtering to days that "balance", where charge and discharge nearly match. That
selects on the very ratio being measured and returns 99% for every unit. The filter is the answer.

**The second trap is idle draw.** Counting every negative interval as charging folds auxiliary
consumption into the denominator. Both figures are plotted; for six of seven units they differ by
0.1 pp or less, because auxiliary load is 0.1-0.5 GWh against 200-400 GWh of throughput.

The fleet lands at **86-88%**, exactly where lithium grid storage should. The right panel is the
degradation check: Kwinana ESR1 has twelve quarters of record, and after its first commissioning
quarter its efficiency does not fall — it drifts *upward*, from 84% in early 2024 to 86-88% since.
Three years is short for a degradation study and the quarterly bound is wider than the annual one,
so read this as "no decline detectable yet" rather than as a clean bill of health.

In [9]:
# ── Chart 4: round-trip efficiency, with the SoC-drift error bound ───────────
rows = []
for f in FACS:
    g = s[(s.facility_code == f) & (s.ts >= mature[f])]
    C_all, D_all = g.chg.sum(), g.dis.sum()
    d_ = g[g.mw.abs() > DISPATCH_MW]
    C_dis, D_dis = d_.chg.sum(), d_.dis.sum()
    rows.append(dict(
        facility=f, months=len(g) / 8640, charge_GWh=C_all / 1e3, dis_GWh=D_all / 1e3,
        rte_all=D_all / C_all, rte_dispatch=D_dis / C_dis, aux_GWh=(C_all - C_dis) / 1e3,
        # Worst-case error: the unit could have started empty and ended full, or the
        # reverse. That is one capacity's worth of energy, over the charge total.
        bound_pp=100 * cap.loc[f, "p95"] / C_all))
rte = pd.DataFrame(rows).set_index("facility").reindex(FACS)

ok = rte.drop(index="ALINTA_WGP_ESR1", errors="ignore")
order = ok.sort_values("rte_dispatch").index.tolist()

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13, column_widths=[0.62, 0.38],
                    subplot_titles=["Round-trip efficiency over each unit's mature record",
                                    "KWINANA_ESR1, quarter by quarter"])

fig.add_trace(go.Scatter(
    x=ok.loc[order, "rte_all"], y=order, mode="markers", name="including idle auxiliary draw",
    marker=dict(size=13, color="rgba(0,0,0,0)", line=dict(color=MUTED, width=1.6)),
    hovertemplate="%{y}<br>%{x:.1%} including auxiliary<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(
    x=ok.loc[order, "rte_dispatch"], y=order, mode="markers", name="dispatch intervals only",
    marker=dict(size=10, color=BLUE, line=dict(color=SURFACE, width=2)),
    error_x=dict(type="data", array=ok.loc[order, "bound_pp"] / 100, color=AXIS,
                 thickness=1.4, width=4),
    hovertemplate="%{y}<br>%{x:.1%} dispatch only<extra></extra>"), row=1, col=1)

# Quarterly, not annual: three years give only two complete calendar years, and
# two points cannot show a trend. A quarter is short enough that boundary drift
# matters, so its bound is drawn too — under 1.1 pp everywhere after commissioning.
ky = s[(s.facility_code == "KWINANA_ESR1") & (s.ts >= mature["KWINANA_ESR1"])].copy()
ky["q"] = ky.ts.dt.to_period("Q").dt.to_timestamp()
qd = ky[ky.mw.abs() > DISPATCH_MW].groupby("q").agg(chg=("chg", "sum"), dis=("dis", "sum"))
qd["rte"] = qd.dis / qd.chg
qd["bound"] = cap.loc["KWINANA_ESR1", "p95"] / qd.chg
fig.add_trace(go.Scatter(
    x=qd.index, y=qd.rte, mode="lines+markers", showlegend=False,
    line=dict(color=BLUE, width=2), marker=dict(size=8, line=dict(color=SURFACE, width=1.5)),
    error_y=dict(type="data", array=qd.bound, color=AXIS, thickness=1.2, width=3),
    hovertemplate="%{x|%b %Y}<br>%{y:.1%}<extra>KWINANA_ESR1</extra>"), row=1, col=2)

style(fig, "The fleet round-trips at 86 to 88 percent, and is not yet degrading",
      "Discharge divided by charge over each unit's full mature record. The bar is the "
      "worst-case error from unknown start and end state of charge.",
      height=470, hover="closest", top=150, legend_y=1.13)
fig.update_xaxes(title_text="round-trip efficiency", tickformat=".0%",
                 range=[0.845, 0.885], row=1, col=1)
fig.update_yaxes(categoryorder="array", categoryarray=order, row=1, col=1)
fig.update_xaxes(title_text=None, tickformat="%Y", row=1, col=2)
fig.update_yaxes(title_text="round-trip efficiency", tickformat=".0%",
                 range=[0.79, 0.90], row=1, col=2)
fig.show()

print("round-trip efficiency, full mature record per unit:")
print(rte[["months", "charge_GWh", "dis_GWh", "aux_GWh", "rte_all", "rte_dispatch",
           "bound_pp"]].round(3).to_string())
print("\nALINTA_WGP_ESR1 is reported but not plotted: 0.9 months of ramping, and a")
print("7.5 pp error bound, which is to say no usable estimate yet.")
print("\nWhy a month is too short — the same ratio computed month by month:")
for f in FACS:
    g = s[(s.facility_code == f) & (s.ts >= mature[f])]
    m = g.groupby("ym").apply(lambda x: x.dis.sum() / x.chg.sum() if x.chg.sum() else np.nan,
                              include_groups=False).dropna()
    if len(m) < 3:
        continue
    print(f"  {f:<16} {len(m):>2} months   min {m.min():.3f}   max {m.max():.3f}"
          f"   above 1.0: {int((m > 1).sum())}")
print("  Three units return impossible months. Nothing is wrong with them: a month is simply")
print("  too short for the unknown boundary state of charge to average out.")

round-trip efficiency, full mature record per unit:
                 months  charge_GWh  dis_GWh  aux_GWh  rte_all  rte_dispatch  bound_pp
facility                                                                              
KWINANA_ESR1     35.219     298.480  257.932    0.164    0.864         0.863     0.073
COLLIE_ESR1      23.803     368.131  320.591    0.526    0.871         0.871     0.203
KWINANA_ESR2     20.964     401.853  346.110    0.240    0.861         0.861     0.222
COLLIE_BESS2     14.702     321.297  281.762    0.289    0.877         0.877     0.370
COLLIE_ESR4       9.952     214.350  184.601    0.148    0.861         0.860     0.487
COLLIE_ESR5       9.034     228.333  198.119    0.116    0.868         0.867     0.453
ALINTA_WGP_ESR1   0.889       0.866    0.363    0.181    0.419         0.529     7.499

ALINTA_WGP_ESR1 is reported but not plotted: 0.9 months of ramping, and a
7.5 pp error bound, which is to say no usable estimate yet.

Why a month is too short — th

  COLLIE_BESS2     16 months   min 0.013   max 0.907   above 1.0: 0
  COLLIE_ESR4      11 months   min 0.763   max 1.578   above 1.0: 1
  COLLIE_ESR5      10 months   min 0.846   max 1.821   above 1.0: 1
  Three units return impossible months. Nothing is wrong with them: a month is simply
  too short for the unknown boundary state of charge to average out.


### 5. The daily cycle, deepening

Averaging every day of a season onto one 24-hour axis, with the whole fleet summed to a single net
figure: below zero the fleet is a load, above zero it is a generator.

The shape barely changes; the amplitude changes enormously. Winter 2024 is a ripple — a trough of
-20 MW and a peak of 23 MW. Winter 2026 is a trough of -534 MW around midday and a peak of 541 MW at
18:00. That is the same machine cycle scaled twenty-five-fold, and it is the direct counterpart of
the price notebook's finding that the daily price shape flattened from both ends: the fleet buys in
the solar trough and sells into the evening peak, which pushes both toward the middle.

The 2026 curves also show something the earlier years are too small to reveal: the fleet discharges
**twice** a day. A sharp morning peak at 07:25 reaching 425 MW in Winter, then the deep midday
charge, then the larger evening peak. The morning hump is the pre-solar demand peak, and the fleet
only started serving it once there was enough capacity to do both.

Seasons are meteorological and southern-hemisphere, matching the other notebooks, and a
(year, season) pair is plotted only if all three of its months are present.

In [10]:
# ── Chart 5: the average day, by season and year ─────────────────────────────
SEASON_ORDER = ["Summer", "Autumn", "Winter", "Spring"]
SEASON_MONTHS = {"Summer": {12, 1, 2}, "Autumn": {3, 4, 5},
                 "Winter": {6, 7, 8}, "Spring": {9, 10, 11}}
MIN_DAYS = 30

fleet = s.groupby("ts").mw.sum().rename("fleet_mw").reset_index()
fleet = add_time_parts(fleet)

have = fleet.groupby(["year", "season"]).agg(days=("date", "nunique"),
                                             months=("month", lambda x: set(x.unique())))
keep = {(y, ss) for (y, ss), row in have.iterrows()
        if row.days >= MIN_DAYS and row.months == SEASON_MONTHS[ss]}
print("plotted:", sorted(keep))
print("dropped:", sorted(set(have.index) - keep))

prof = fleet.groupby(["year", "season", "tod_min"]).fleet_mw.mean().reset_index()

fig = make_subplots(rows=1, cols=4, shared_yaxes=True, horizontal_spacing=0.025,
                    subplot_titles=SEASON_ORDER)
seen = set()
for i, ss in enumerate(SEASON_ORDER, start=1):
    for y in YEARS:
        if (y, ss) not in keep:
            continue
        g = prof[(prof.year == y) & (prof.season == ss)].sort_values("tod_min")
        fig.add_trace(go.Scatter(
            x=g.tod_min / 60, y=g.fleet_mw, name=LABEL[y], legendgroup=str(y),
            showlegend=y not in seen,
            line=dict(color=YEAR_COLOR[y], width=2, dash=DASH[y]),
            hovertemplate="%{x:.1f}h · %{y:,.0f} MW" f"<extra>{LABEL[y]} {ss}</extra>"),
            row=1, col=i)
        seen.add(y)
    fig.add_hline(y=0, line=dict(color=AXIS, width=1), row=1, col=i)

style(fig, "The same cycle, twenty-five times deeper",
      "Fleet net output by time of day. Below zero the fleet is charging; above zero "
      "it is discharging. Only seasons with all three months present are shown.",
      height=470, top=150, legend_y=1.14)
fig.update_xaxes(tickvals=[0, 6, 12, 18, 24], range=[0, 24], title_text="hour")
fig.update_yaxes(title_text="fleet net MW", row=1, col=1)
fig.show()

print("Winter trough and peak, by year:")
for y in (2024, 2025, 2026):
    g = prof[(prof.year == y) & (prof.season == "Winter")]
    if not len(g):
        continue
    lo, hi = g.loc[g.fleet_mw.idxmin()], g.loc[g.fleet_mw.idxmax()]
    print(f"  {y}:  trough {lo.fleet_mw:7.1f} MW at {int(lo.tod_min)//60:02d}:00"
          f"   peak {hi.fleet_mw:7.1f} MW at {int(hi.tod_min)//60:02d}:00"
          f"   swing {hi.fleet_mw - lo.fleet_mw:7.1f} MW")

plotted: [(2024, 'Autumn'), (2024, 'Spring'), (2024, 'Summer'), (2024, 'Winter'), (2025, 'Autumn'), (2025, 'Spring'), (2025, 'Summer'), (2025, 'Winter'), (2026, 'Autumn'), (2026, 'Winter')]
dropped: [(2023, 'Spring'), (2023, 'Summer'), (2026, 'Spring'), (2026, 'Summer')]


Winter trough and peak, by year:
  2024:  trough   -19.8 MW at 12:00   peak    23.0 MW at 07:00   swing    42.8 MW
  2025:  trough  -207.8 MW at 14:00   peak   263.1 MW at 18:00   swing   470.9 MW
  2026:  trough  -534.2 MW at 12:00   peak   541.3 MW at 18:00   swing  1075.5 MW


### 6. Spread capture and the margin collapse

What the fleet actually earned. Every 5-minute interval is aggregated to the 30-minute trading
interval and joined to the reference trading price, giving a volume-weighted **buy price** across all
charging and a volume-weighted **sell price** across all discharging. The difference is spread
capture: gross margin per MWh cycled, before efficiency losses and before any other revenue stream.

The upper panel is those two prices. They start more than 100 $/MWh apart and end almost touching.
The buy side is the half that moved: the fleet charged at **31 $/MWh** in 2024 and pays
**107 $/MWh** by 2026, while the sell price fell only from 145 to 127. Collie ESR1, the only large
unit running for most of 2024, averaged **-26 $/MWh** to charge that year — it was paid to take
energy, in the midday negative-price block the price notebook mapped. That block is gone, and the
fleet ate it.

The lower panel is the consequence, and it is the sharpest result in this notebook. Energy-only gross
margin peaked in 2025 at **25.4 million dollars** and fell to **3.2 million** in 2026 — while
discharged volume grew from 605 to 837 GWh, up 38%. More throughput, less money. Five of the seven
units returned a **negative** energy margin in 2026, and only Collie ESR1 and Collie BESS2 cleared
one million.

Read that as an energy-market result and nothing more. Capacity credits and essential-system-service
payments are real revenue for these units and are not in this dataset. The claim here is narrow and
well supported: **the energy-arbitrage case has closed**, and it closed while the fleet was still
being built.

In [11]:
# ── Chart 6: buy price, sell price, and what is left ─────────────────────────
price = load_price(raw='../data/raw')
ti = s.assign(ti=s.ts.dt.floor("30min")).groupby(["facility_code", "ti"])[["chg", "dis"]].sum()
ti = ti.reset_index().merge(price[["ts", "price"]].rename(columns={"ts": "ti"}),
                            on="ti", how="left")
assert ti.price.isna().sum() == 0, "every trading interval must carry a price"
ti["cost"] = ti.chg * ti.price          # dollars paid to charge
ti["rev"] = ti.dis * ti.price           # dollars earned discharging
ti["ym"] = ti.ti.dt.to_period("M").dt.to_timestamp()
ti["year"] = ti.ti.dt.year

mo = ti[ti.ym.isin(FULL_MONTHS)].groupby("ym").agg(
    chg=("chg", "sum"), dis=("dis", "sum"), cost=("cost", "sum"), rev=("rev", "sum"))
mo["buy"] = mo.cost / mo.chg
mo["sell"] = mo.rev / mo.dis
mo["margin_m"] = (mo.rev - mo.cost) / 1e6

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    row_heights=[0.58, 0.42],
                    subplot_titles=["Volume-weighted buy and sell price",
                                    "Energy-only gross margin per month"])

# The band between the two lines IS the spread, so it is filled rather than
# annotated: the quantity of interest is the gap, and the gap closing is the story.
fig.add_trace(go.Scatter(x=mo.index, y=mo.sell, name="sell price (discharging)",
                         line=dict(color=BLUE, width=2),
                         hovertemplate="%{y:,.0f} $/MWh<extra>sell</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=mo.index, y=mo.buy, name="buy price (charging)",
                         line=dict(color=ORANGE, width=2), fill="tonexty",
                         fillcolor="rgba(42,120,214,0.12)",
                         hovertemplate="%{y:,.0f} $/MWh<extra>buy</extra>"), row=1, col=1)
fig.add_hline(y=0, line=dict(color=AXIS, width=1, dash="dot"), row=1, col=1)
for xx, lab in ((pd.Timestamp("2024-06-01"), "spread"),):
    fig.add_annotation(x=xx, y=(mo.loc[xx, "buy"] + mo.loc[xx, "sell"]) / 2, text=lab,
                       showarrow=False, font=dict(size=11, color=MUTED), row=1, col=1)

col = [BLUE if v >= 0 else RED for v in mo.margin_m]
fig.add_trace(go.Bar(x=mo.index, y=mo.margin_m, marker=dict(color=col, line_width=0),
                     showlegend=False, width=22 * 86400000,
                     hovertemplate="%{x|%b %Y}<br>%{y:.2f} million dollars<extra></extra>"),
              row=2, col=1)
fig.add_hline(y=0, line=dict(color=AXIS, width=1), row=2, col=1)

style(fig, "The gap closed, and the margin went with it",
      "Volume-weighted prices the fleet charged and discharged at, and the gross energy "
      "margin that gap produced. Energy arbitrage only.",
      height=620, top=150, legend_y=1.10)
fig.update_yaxes(title_text="$/MWh", row=1, col=1)
fig.update_yaxes(title_text="million dollars", row=2, col=1)
fig.show()

yr = ti.groupby("year").agg(chg=("chg", "sum"), dis=("dis", "sum"),
                            cost=("cost", "sum"), rev=("rev", "sum"))
yr["buy"] = yr.cost / yr.chg
yr["sell"] = yr.rev / yr.dis
yr["spread"] = yr.sell - yr.buy
yr["GWh_out"] = yr.dis / 1e3
yr["gross_m"] = (yr.rev - yr.cost) / 1e6
print("fleet energy arbitrage, by year (2023 and 2026 are partial years):")
print(yr[["buy", "sell", "spread", "GWh_out", "gross_m"]].round(2).to_string())

pf = ti[ti.year == 2026].groupby("facility_code").apply(
    lambda g: (g.rev.sum() - g.cost.sum()) / 1e6, include_groups=False).reindex(FACS)
print("\n2026 energy-only gross margin by facility (million dollars):")
print(pf.round(2).to_string())
print(f"\n{int((pf < 0).sum())} of {pf.notna().sum()} units negative on energy arbitrage in 2026.")

fleet energy arbitrage, by year (2023 and 2026 are partial years):
         buy    sell  spread  GWh_out  gross_m
year                                          
2023   23.74  149.18  125.44     4.72     0.57
2024   30.57  145.02  114.45   149.28    16.34
2025   62.00  113.69   51.69   604.71    25.35
2026  106.68  126.81   20.13   836.74     3.19

2026 energy-only gross margin by facility (million dollars):
facility_code
KWINANA_ESR1      -1.02
COLLIE_ESR1        2.86
KWINANA_ESR2      -0.49
COLLIE_BESS2       4.36
COLLIE_ESR4       -0.90
COLLIE_ESR5       -1.55
ALINTA_WGP_ESR1   -0.08

5 of 7 units negative on energy arbitrage in 2026.


## Comparability checks

Four things could each have manufactured the results above.

1. **The partial years.** 2026 runs only to 1 September and is missing November and December. Every
   year-on-year claim is re-checked on a like-for-like January-August basis.
2. **Fleet growth.** Total margin falling while the fleet grows is a stronger claim than it looks, so
   margin is also expressed per MW of installed power — the per-unit economics, not the aggregate.
3. **Energy only.** The margin figures exclude capacity credits and essential system services. This
   cannot be tested with this data; it is stated as a limit on the claim, and the AEMO
   `capacity-credits-since-market-start` and `fcess` datasets are where it would be tested.
4. **The demand identity.** `docs/DATA_NOTES.md` quirk 5 treats `Operational Withdrawal` as
   grid-scale storage charging. Summing this fleet's charging against that column tests it directly
   — and it does not hold exactly.

In [12]:
# ── Comparability checks ─────────────────────────────────────────────────────
ja = ti[ti.ti.dt.month <= 8]
g = ja.groupby("year").agg(chg=("chg", "sum"), dis=("dis", "sum"),
                           cost=("cost", "sum"), rev=("rev", "sum"))
g["buy"] = g.cost / g.chg
g["sell"] = g.rev / g.dis
g["spread"] = g.sell - g.buy
g["GWh_out"] = g.dis / 1e3
g["gross_m"] = (g.rev - g.cost) / 1e6
print("1. LIKE-FOR-LIKE (January-August only) — the collapse survives:")
print(g[["buy", "sell", "spread", "GWh_out", "gross_m"]].round(2).to_string())

print("\n2. PER MW OF INSTALLED POWER (thousand dollars per MW, Jan-Aug) —")
print("   this is not an aggregation artefact; the per-unit economics fall too:")
pm = (ja.groupby(["facility_code", "year"])
      .apply(lambda d: (d.rev.sum() - d.cost.sum()), include_groups=False)
      .rename("gm").reset_index())
pm["k_per_MW"] = pm.gm / pm.facility_code.map(PMAX) / 1e3
print(pm.pivot(index="facility_code", columns="year", values="k_per_MW")
      .reindex(FACS).round(1).to_string())

print("\n3. ENERGY ONLY — untestable here, and the single largest limit on the result above.")
print("   Capacity credits and ESS payments are not in this dataset.")

print("\n4. THE DEMAND IDENTITY — quirk 5 does not hold exactly.")
dem = load_demand(raw='../data/raw').set_index("ts")
fleet_chg = s.assign(c=np.where(s.mw < 0, s.mw, 0.0)).groupby("ts").c.sum()
chk = dem.join(fleet_chg.rename("scada_chg"), how="left")
chk["resid"] = chk.withdrawal_mw - chk.scada_chg
print(f"   withdrawal_mw minus this fleet's charging: mean {chk.resid.mean():.2f} MW, "
      f"median {chk.resid.median():.2f} MW")
print("   Withdrawal is SYSTEMATICALLY larger than storage charging, in every year:")
print(chk.groupby(chk.index.year).resid.mean().round(2).to_string())

# The decisive case: intervals where withdrawal exceeds what the whole fleet could
# physically draw, given which units existed at the time.
capacity = pd.Series(0.0, index=pd.Index(sorted(s.ts.unique()), name="ts"))
for f in FACS:
    capacity += np.where(capacity.index >= first[f], -s[s.facility_code == f].mw.min(), 0.0)
imp = chk.join(capacity.rename("fleet_cap"))
imp = imp[imp.withdrawal_mw < -(imp.fleet_cap * 1.1 + 20)]
print(f"\n   intervals where |withdrawal| exceeds the ENTIRE fleet's charging capability: "
      f"{len(imp)} of {len(chk):,}")
print(imp[["operational_demand_mw", "unscheduled_demand_mw", "withdrawal_mw",
           "fleet_cap"]].round(1).to_string())
print("   Both are 7 December 2023, when the fleet was one 100 MW battery, and both revert")
print("   within one interval while operational demand barely moves: they are bad intervals")
print("   in OperationalDemandWithdrawal, not real withdrawal.")
print("\n   The systematic residual is a different matter and it is real. Checked against the")
print("   unfiltered FacilityScada-2023-12.csv (28 MB, not kept in data/raw), the gap closes:")
print("   withdrawal minus THIS fleet's charging averages -5.67 MW that month, but withdrawal")
print("   minus EVERY facility's negative output averages -0.17 MW, median -0.01. Gas turbines")
print("   draw real power — KEMERTON_GT11 and GT12 reach -203 MW each — and dozens of")
print("   generators carry small house loads. `Operational Withdrawal` is total withdrawal by")
print("   all facilities, NOT storage charging alone, and quirk 5 needs that qualification.")

1. LIKE-FOR-LIKE (January-August only) — the collapse survives:
         buy    sell  spread  GWh_out  gross_m
year                                          
2024   45.18  168.56  123.38    55.59     6.42
2025   74.51  129.49   54.99   315.05    13.53
2026  106.69  126.85   20.16   835.61     3.15

2. PER MW OF INSTALLED POWER (thousand dollars per MW, Jan-Aug) —
   this is not an aggregation artefact; the per-unit economics fall too:
year             2024  2025  2026
facility_code                    
KWINANA_ESR1     64.6   4.8 -10.1
COLLIE_ESR1      -0.4  35.0  14.3
KWINANA_ESR2      NaN  16.4  -2.2
COLLIE_BESS2      NaN   7.8  14.6
COLLIE_ESR4       NaN   NaN  -3.7
COLLIE_ESR5       NaN   NaN  -6.2
ALINTA_WGP_ESR1   NaN   NaN  -3.0

3. ENERGY ONLY — untestable here, and the single largest limit on the result above.
   Capacity credits and ESS payments are not in this dataset.

4. THE DEMAND IDENTITY — quirk 5 does not hold exactly.


   withdrawal_mw minus this fleet's charging: mean -7.60 MW, median -5.97 MW
   Withdrawal is SYSTEMATICALLY larger than storage charging, in every year:
ts
2023   -6.37
2024   -6.42
2025   -8.06
2026   -9.14



   intervals where |withdrawal| exceeds the ENTIRE fleet's charging capability: 2 of 307,008
                     operational_demand_mw  unscheduled_demand_mw  withdrawal_mw  fleet_cap
ts                                                                                         
2023-12-07 13:15:00                 1318.9                  671.8         -647.0      100.1
2023-12-07 13:25:00                 1272.0                  354.0         -918.1      100.1
   Both are 7 December 2023, when the fleet was one 100 MW battery, and both revert
   within one interval while operational demand barely moves: they are bad intervals
   in OperationalDemandWithdrawal, not real withdrawal.

   The systematic residual is a different matter and it is real. Checked against the
   unfiltered FacilityScada-2023-12.csv (28 MB, not kept in data/raw), the gap closes:
   withdrawal minus THIS fleet's charging averages -5.67 MW that month, but withdrawal
   minus EVERY facility's negative output averages -0

## What this notebook establishes

- **The fleet is seven batteries, 1,357 MW of discharge power, and it is almost all new.** One unit
  in October 2023, five more between July 2024 and November 2025, and Alinta Wagerup in July 2026.
  Discharged energy went from 4.7 GWh in 2023 to 837 GWh in the first eight months of 2026.
- **Six of the seven are 4-hour batteries; Kwinana ESR1 is a 2-hour battery.** Neither fact is in the
  registry. Both come out of the dispatch record by integrating charge and discharge into a relative
  state of charge and reading off the daily swing — 218 MWh against 100 MW for Kwinana ESR1, 748 to
  1,190 MWh against 200 to 300 MW for the rest.
- **Round-trip efficiency is 86-88% and flat.** Measured over each unit's full mature record, where
  the unknown boundary state of charge contributes under 0.5 pp of error. Kwinana ESR1 shows no
  degradation across three years. Auxiliary draw is real but immaterial to the ratio.
- **The fleet does one thing, and does it harder every year.** Charge midday, discharge into the
  evening peak. The mean Winter daily swing went from 43 MW peak-to-trough in 2024 to 1,075 MW in
  2026.
- **Energy arbitrage no longer pays for these machines.** Spread capture fell from 123 to 20 $/MWh
  between 2024 and 2026 on a like-for-like January-August basis, and gross energy margin from a 2025
  peak of 25.4 million dollars to 3.2 million in 2026, on 38% more volume. Five of seven units are
  energy-negative in 2026. The buy side did the damage: the fleet charged at 31 $/MWh in 2024 and at
  107 $/MWh in 2026, while the price it sold into fell far less.

That last point answers the question the price notebook left open, and it answers it in the
direction the price notebook suspected but could not show. The flattening of the daily price shape
and the collapse of the arbitrage spread are the same event seen from two sides: the fleet bought
the trough until the trough went away. **Storage did not merely arrive alongside the flattening; it
is the mechanism.** Two series moving oppositely was weak evidence, but volume-weighted buy and sell
prices converging on each other, driven by the buy side, taken from the batteries' own dispatch, is
not.

Two limits on all of this, and one correction to the documented data notes:

- **Energy only.** Capacity credits and essential system services are not in this dataset and are
  material to whether these assets are profitable. Nothing here says a battery is a bad investment;
  it says the energy-market half of the case has closed.
- **Five weeks of Alinta Wagerup is not a sample.** It began dispatching 25 July 2026 at an eighth of
  its registered size and is excluded from every fleet statistic that is not a simple sum.
- **`Operational Withdrawal` is not storage charging.** It is withdrawal by all facilities, including
  gas-turbine auxiliary load of up to 203 MW. Quirk 5 in `docs/DATA_NOTES.md` should be qualified,
  and the two corrupt intervals of 7 December 2023 recorded alongside it.